In [73]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.inspection import permutation_importance

from ml_enhance import custom_train_test_split, load_hpc_result

def filter_X(X: pd.DataFrame) -> pd.DataFrame:
    drop_qm_features = [
        "radius_of_gyration",
        "molecular_volume",
        "sterimol_L",
        "sterimol_Bmin",
        "sterimol_Bmax",
        "molecular_sasa",
        "solvation_energy_thf",
        "solvation_energy_cyclohexane",
        "solvation_energy_dmso",
        "avg_percentage_buried_volume",
        "min_percentage_buried_volume",
        "max_percentage_buried_volume",
        "std_percentage_buried_volume",
        "avg_atomic_sasa",
        "min_atomic_sasa",
        "max_atomic_sasa",
        "std_atomic_sasa",
        "min_partial_charge_thf",
        "max_partial_charge_thf",
        "std_partial_charge_thf",
        "min_partial_charge_cyclohexane",
        "max_partial_charge_cyclohexane",
        "std_partial_charge_cyclohexane",
        "min_partial_charge_dmso",
        "max_partial_charge_dmso",
        "std_partial_charge_dmso",
        "avg_bond_length",
        "min_bond_length",
        "max_bond_length",
        "std_bond_length",
        "avg_bond_stiffness",
        "min_bond_stiffness",
        "max_bond_stiffness",
        "std_bond_stiffness",
        # "avg_effective_coordination_number",
        # "min_effective_coordination_number",
        # "max_effective_coordination_number",
        # "std_effective_coordination_number",
    ]

    drop_topo_features = [
        "MaxPartialCharge",
        "MinPartialCharge",
        "MaxAbsPartialCharge",
        "MinAbsPartialCharge",
    ]

    return X.drop(
        [
            "avg_atomic_quadrupole_principal_invariant_3",  # quadrupole principal invariant 3 features correlate highly with the invariant 2 features, so can drop them
            "max_atomic_quadrupole_principal_invariant_3",
            "molecular_quadrupole_principal_invariant_3",
            "avg_atomic_dipole_dipole_interaction",  # the dipole dipole interaction between atoms would physically not be that influential on the solubility, can drop it
        ]
        + drop_qm_features
        + drop_topo_features,
        axis=1,
    )



file_path = Path("data/HuberReg_results/HuberReg_combo_pure.pkl")
fold_id = 11

splits_file = Path("hpc_splits.pkl")
rdkit_file = Path("data/rdkit_feature_names.json")

df = pd.read_csv("data/processed_dataset_wo_metals_w_even_more_qm2.csv")
X = df.drop(["solubility", "smiles", "canon_smiles", "id"], axis=1)
X_filtered = filter_X(X)
y = df["solubility"]

model_df: pd.DataFrame = load_hpc_result(file_path, file_path.stem)
model = model_df["estimator"][fold_id]

_, X_test, _, y_test = custom_train_test_split(splits_file, fold_id, X_filtered, y)

scoring = {"r2": "r2", "MSE": "neg_mean_squared_error"}

PFI = permutation_importance(model.best_estimator_, X_test, y_test, scoring=scoring, n_repeats=20, random_state=9, n_jobs=5)

In [79]:
from ml_enhance import plot_FI


df = pd.concat(
    {
        metric: pd.DataFrame({
            "mean": r.importances_mean,
            "std": r.importances_std
        })
        for metric, r in PFI.items()
    },
    axis=1
)
df.index = X_test.columns
df.columns = [f"{metric}_{stat}" for metric, stat in df.columns]
# plot_FI(df['r2_mean'], 20)

In [95]:
features = model.best_estimator_[:-1].get_feature_names_out()
features
# PFI["r2"]["importances_mean"].shape

array(['atomization_energy', 'homo_lumo_gap', 'ionization_energy',
       'electron_affinity', 'chemical_potential', 'molecular_dipole_norm',
       'molecular_quadrupole_principal_invariant_2',
       'molecular_polarizability_mean',
       'molecular_polarizability_anisotropy', 'solvation_energy_water',
       'delta_gibbs_free_energy_300K', 'gibbs_free_energy_300K_range',
       'delta_enthalpy', 'delta_entropy_300K', 'entropy_300K_range',
       'std_entropy_300K', 'delta_heat_capacity_300K',
       'heat_capacity_300K_range', 'std_heat_capacity_300K', 'rigid_flag',
       'ir_centroid_freq_1500', 'ir_norm_intensity_1500',
       'ir_mode_count_1500_2750', 'ir_centroid_freq_1500_2750',
       'ir_norm_intensity_1500_2750', 'ir_centroid_freq_2750_4000',
       'ir_norm_intensity_2750_4000', 'avg_effective_coordination_number',
       'min_effective_coordination_number',
       'max_effective_coordination_number',
       'std_effective_coordination_number', 'avg_atomic_dipole_norm',


In [81]:
X_test.columns.shape

(308,)

In [82]:
X.filter(regex="Log|logP").columns

Index(['SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12', 'SlogP_VSA2',
       'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA6', 'SlogP_VSA7',
       'SlogP_VSA8', 'SlogP_VSA9', 'MolLogP'],
      dtype='str')

In [89]:
b = df.index

b[~b.isin(features)].shape

(52,)

In [88]:
df.shape

(308, 4)

In [98]:
df[df["r2_mean"] == 0.0].index[df[df["r2_mean"] == 0.0].index.isin(features)]

Index(['min_effective_coordination_number', 'fr_diazo', 'fr_dihydropyridine',
       'fr_thiocyan'],
      dtype='str')